# Phase 4 — Validation-Selected Test Ablation

Freezes the best Phase 2 prompt and available Phase 3 Track A configuration using development data,
then loads the target test split exactly once for the final four-condition Gemma ablation.


## 1. Environment


In [ ]:
# Phase 4 uses the Gemma-compatible runtime.
!pip install -q -U "transformers==5.14.1" datasets bitsandbytes accelerate statsmodels

from google.colab import drive
drive.mount("/content/drive")

# ── Published benchmark reference ───────────
BENCHMARK_A = {
    "eng": 0.823, "hin": 0.926, "rus": 0.901, "hau": 0.751, "kin": 0.657,
    "sun": 0.550, "yor": 0.461, "vmw": 0.325, "pcm": 0.674,
}
BENCHMARK_C = {
    "eng": 0.797, "hin": 0.919, "rus": 0.906, "hau": 0.709, "kin": 0.519,
    "sun": 0.467, "yor": 0.359, "vmw": 0.210, "pcm": 0.674,
}
BENCHMARK_STATUS = (
    "Descriptive context only: Phase 4 is an internal CAST ablation, "
    "not an official Track A/C submission"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 48.9 MB/s eta 0:00:00
Mounted at /content/drive


## 2. Configuration and scoring


In [ ]:
import gc
import json
import os
import random

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import f1_score
from statsmodels.stats.multitest import multipletests


PROJECT_ROOT = "/content/drive/MyDrive/EmotionDetection"
VALID_CKPT = f"{PROJECT_ROOT}/CAST_checkpoints"
P2_DIR,P3_DIR,P4_DIR = [f"{VALID_CKPT}/Phase{i}" for i in (2,3,4)]
VALIDATION_PRED_DIR,PHASE3_VALIDATION_PRED_DIR = f"{P2_DIR}/predictions",f"{P3_DIR}/predictions"
TEST_PRED_DIR,SELECTION_PATH = f"{P4_DIR}/predictions",f"{P4_DIR}/phase4_validation_selections.json"
PILOT_DIR = P4_DIR

LANG_ORDER = ["eng","hin","rus","hau","kin","sun","yor","vmw","pcm"]
RUN_LANGS = LANG_ORDER.copy()
RUN_TEST_INFERENCE = True
GEMMA_MODEL,XLMR_MODEL_NAME = "unsloth/gemma-4-31B-it-unsloth-bnb-4bit","xlm-roberta-large"
BATCH_SIZE,FEW_SHOT_K,RANDOM_SEED,N_BOOT = 8,2,42,10_000

PROMPT_ORDER,CONFIG_ORDER = ["p0","p1","p2","p3"],["C1","C2","C3"]
EMOTION_ORDER = ["anger","disgust","fear","joy","sadness","surprise"]
ALL_TARGET_CODES = LANG_ORDER.copy()
ABSENT_EMOTIONS = {"eng":{"disgust"}}

for path in [P4_DIR,TEST_PRED_DIR]: os.makedirs(path,exist_ok=True)

CACHE_DIR,MODEL_CACHE_DIR = f"{P3_DIR}/data_cache",f"{PROJECT_ROOT}/model_cache"
LOCAL_MODEL_CACHE="/content/hf_cache_local"

for path in [MODEL_CACHE_DIR,LOCAL_MODEL_CACHE]: os.makedirs(path,exist_ok=True)
os.environ.update(HF_HOME=MODEL_CACHE_DIR,HF_DATASETS_CACHE=f"{MODEL_CACHE_DIR}/datasets",
                  HF_HUB_DISABLE_SYMLINKS_WARNING="1")

GENERATION_CONFIG={"max_new_tokens":30,"do_sample":False}

LANGUAGES = {
    "eng":{"name":"English","tier":1,"family":"Indo-European","genus":"Germanic"},
    "hin":{"name":"Hindi","tier":1,"family":"Indo-European","genus":"Indo-Aryan"},
    "rus":{"name":"Russian","tier":1,"family":"Indo-European","genus":"Slavic"},
    "hau":{"name":"Hausa","tier":2,"family":"Afroasiatic","genus":"Chadic"},
    "kin":{"name":"Kinyarwanda","tier":2,"family":"Niger-Congo","genus":"Bantu"},
    "sun":{"name":"Sundanese","tier":2,"family":"Austronesian","genus":"Sundic"},
    "yor":{"name":"Yoruba","tier":3,"family":"Niger-Congo","genus":"Volta-Niger"},
    "vmw":{"name":"Emakhuwa","tier":3,"family":"Niger-Congo","genus":"Bantu"},
    "pcm":{"name":"Nigerian Pidgin","tier":3,"family":"Creole","genus":"English-Lexifier"},
}
EXTRA_LANGS = {
    "mar":{"family":"Indo-European","genus":"Indo-Aryan"},
    "ukr":{"family":"Indo-European","genus":"Slavic"},
    "esp":{"family":"Indo-European","genus":"Romance"},
    "ptbr":{"family":"Indo-European","genus":"Romance"},
    "ptmz":{"family":"Indo-European","genus":"Romance"},
    "ron":{"family":"Indo-European","genus":"Romance"},
    "deu":{"family":"Indo-European","genus":"Germanic"},
    "swe":{"family":"Indo-European","genus":"Germanic"},
    "afr":{"family":"Indo-European","genus":"Germanic"},
    "arq":{"family":"Afroasiatic","genus":"Semitic"},
    "ary":{"family":"Afroasiatic","genus":"Semitic"},
    "swa":{"family":"Niger-Congo","genus":"Bantu"},
    "ibo":{"family":"Niger-Congo","genus":"Volta-Niger"},
}

def atomic_json_write(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temporary = f"{path}.tmp"
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def load_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

def active_emotions(language):
    absent = ABSENT_EMOTIONS.get(language, set())
    return [emotion for emotion in EMOTION_ORDER if emotion not in absent]

def labels_to_matrix(frame):
    matrix = np.zeros((len(frame), len(EMOTION_ORDER)), dtype=int)
    for index, emotion in enumerate(EMOTION_ORDER):
        if emotion in frame.columns:
            matrix[:, index] = frame[emotion].fillna(0).astype(int).to_numpy()
    return matrix

def score_predictions(y_true, y_pred, language):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"{language}: y_true={y_true.shape}, y_pred={y_pred.shape}")
    raw = {
        emotion: float(f1_score(y_true[:, i], y_pred[:, i], zero_division=0))
        for i, emotion in enumerate(EMOTION_ORDER)
    }
    result = {emotion: round(value, 4) for emotion, value in raw.items()}
    result["macro_f1_raw"] = float(np.mean([raw[e] for e in active_emotions(language)]))
    result["macro_f1"] = round(result["macro_f1_raw"], 4)
    return result

print(f"Phase 4 output: {P4_DIR}")
print(f"Languages: {RUN_LANGS}")


Phase 4 output: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase4
Languages: ['eng', 'hin', 'rus', 'hau', 'kin', 'sun', 'yor', 'vmw', 'pcm']


## 3. Transfer pools


In [ ]:
FAMILY = {
    code: metadata["family"]
    for code, metadata in {**LANGUAGES, **EXTRA_LANGS}.items()
}
GENUS = {
    code: metadata["genus"]
    for code, metadata in {**LANGUAGES, **EXTRA_LANGS}.items()
}

PCM_C2 = ["eng", "deu", "swe", "afr"]
PCM_C3 = ["eng"]


def get_c1_pool(target):
    return [code for code in ALL_TARGET_CODES if code != target]


def get_c2_pool(target):
    if target == "pcm":
        return PCM_C2.copy()
    family = FAMILY[target]
    return [
        code for code in FAMILY
        if code != target and FAMILY[code] == family
    ]


def get_c3_pool(target):
    if target == "pcm":
        return PCM_C3.copy()
    if target in ("hau", "sun"):
        return get_c2_pool(target)
    genus = GENUS[target]
    return [
        code for code in GENUS
        if code != target and GENUS[code] == genus
    ]


POOL_FUNCTIONS = {
    "C1": get_c1_pool,
    "C2": get_c2_pool,
    "C3": get_c3_pool,
}

print("Candidate source pools:")
for language in RUN_LANGS:
    print(f"\n{LANGUAGES[language]['name']}:")
    for config in CONFIG_ORDER:
        print(f"  {config}: {POOL_FUNCTIONS[config](language)}")


Candidate source pools:

English:
  C1: ['hin', 'rus', 'hau', 'kin', 'sun', 'yor', 'vmw', 'pcm']
  C2: ['hin', 'rus', 'mar', 'ukr', 'esp', 'ptbr', 'ptmz', 'ron', 'deu', 'swe', 'afr']
  C3: ['deu', 'swe', 'afr']

Hindi:
  C1: ['eng', 'rus', 'hau', 'kin', 'sun', 'yor', 'vmw', 'pcm']
  C2: ['eng', 'rus', 'mar', 'ukr', 'esp', 'ptbr', 'ptmz', 'ron', 'deu', 'swe', 'afr']
  C3: ['mar']

Russian:
  C1: ['eng', 'hin', 'hau', 'kin', 'sun', 'yor', 'vmw', 'pcm']
  C2: ['eng', 'hin', 'mar', 'ukr', 'esp', 'ptbr', 'ptmz', 'ron', 'deu', 'swe', 'afr']
  C3: ['ukr']

Hausa:
  C1: ['eng', 'hin', 'rus', 'kin', 'sun', 'yor', 'vmw', 'pcm']
  C2: ['arq', 'ary']
  C3: ['arq', 'ary']

Kinyarwanda:
  C1: ['eng', 'hin', 'rus', 'hau', 'sun', 'yor', 'vmw', 'pcm']
  C2: ['yor', 'vmw', 'swa', 'ibo']
  C3: ['vmw', 'swa']

Sundanese:
  C1: ['eng', 'hin', 'rus', 'hau', 'kin', 'yor', 'vmw', 'pcm']
  C2: []
  C3: []

Yoruba:
  C1: ['eng', 'hin', 'rus', 'hau', 'kin', 'sun', 'vmw', 'pcm']
  C2: ['kin', 'vmw', 'swa', 'ibo']

## 4. Source-language training data


In [ ]:
def load_split(language, split):
    cache_path = f"{CACHE_DIR}/{language}_{split}.parquet"
    if os.path.exists(cache_path):
        frame = pd.read_parquet(cache_path)
    else:
        dataset = load_dataset("brighter-dataset/BRIGHTER-emotion-categories", language)
        aliases = {"validation": ["validation", "dev"], "train": ["train"], "test": ["test"]}
        split_key = next((key for key in aliases[split] if key in dataset), None)
        if split_key is None:
            raise KeyError(f"{language}: no {split} split; available={list(dataset)}")
        frame = dataset[split_key].to_pandas()
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        frame.to_parquet(cache_path, index=False)
    for emotion in EMOTION_ORDER:
        if emotion not in frame.columns:
            frame[emotion] = 0
    if "text" not in frame.columns:
        raise ValueError(f"{language}/{split}: text column is missing")
    return frame.reset_index(drop=True)

required_sources = set()
for target in RUN_LANGS:
    for config in CONFIG_ORDER:
        required_sources.update(POOL_FUNCTIONS[config](target))

SOURCE_TRAIN_DATA = {}
failures = []
for source in sorted(required_sources):
    try:
        SOURCE_TRAIN_DATA[source] = load_split(source, "train")
    except Exception as exc:
        failures.append((source, str(exc)))
if failures:
    raise RuntimeError(
        "Required source-language training data could not be loaded:\n"
        + "\n".join(f"{source}: {error}" for source, error in failures)
    )
print(f"Loaded {len(SOURCE_TRAIN_DATA)} source-language training sets.")


Loaded 22 source-language training sets.


## 5. Cultural prompts and deterministic few-shot blocks


In [ ]:
CULTURAL_CONTENT = {'eng': {'p1': 'English is a low-context, individualist language where emotions are expressed '
               'directly and individually. The dominant cultural script values emotional '
               'self-control - managing displays is considered a sign of maturity. Positive '
               'emotions are expected openly in public; smiling toward strangers is a default '
               'social norm. Direct verbal labelling is the primary channel - English speakers '
               'name what they feel rather than conveying it indirectly. Individual emotional '
               'autonomy is prioritised over communal regulation. The Anglo cultural ideal '
               'emphasises managing emotion through rational self-direction rather than yielding '
               'to it or suppressing it entirely.',
         'p2': 'Joy: Publicly expressed and socially expected. Smiling toward strangers is '
               'normative. Happiness framed as individual achievement and personal wellbeing '
               'rather than communal experience. Sadness: Expressed directly through verbal '
               'statement. Prolonged public grief is socially uncomfortable - composure is '
               'expected relatively quickly. Sadness framed as temporary and manageable. Anger: '
               'Direct verbal expression culturally acceptable, particularly as assertiveness. '
               'Anger at injustice is legitimate. Aggressive physical displays are socially '
               'sanctioned against. Fear: Acknowledged verbally and directly. Self-disclosure of '
               'vulnerability is acceptable in appropriate contexts - less stigmatised than in '
               'high-context cultures. Surprise: Expressed openly with verbal exclamations. More '
               'restrained in formal contexts, more exuberant in casual speech. Disgust: Primarily '
               'sensory and physical rather than moral - triggered by contamination, bodily '
               'functions, and violations of physical purity. Maps to direct sensory repulsion '
               'more than moral transgression.'},
 'hin': {'p1': 'Hindi is a high-context, collectivist language where emotions are shaped by family '
               'honour, social hierarchy, and communal harmony. Indirect expression is the norm - '
               'conveyed through implication, silence, and bodily metaphors rather than direct '
               'statement. Izzat (honour) functions as a collective family asset; emotions '
               'threatening reputation are suppressed. Lajjā (modesty/shame) is a cultural virtue '
               'acting as public restraint, especially with elders. Collectivist display rules '
               'prioritise social harmony - negative emotions are masked to avoid disrupting group '
               'relations. Low-arousal states like contentment and peace are more valued than '
               'exuberant positivity. Emotions are expressed in Hindi-English code-mixed registers '
               'online.',
         'p2': 'Joy (khushi/sukh): Sukh is deep internal contentment without outward display. '
               'Happiness conceptualised through sweetness, illumination, and moral goodness. Tied '
               'to communal festivals not individual pleasure. Sadness (dukh/gham): Expressed '
               'through dukh (suffering), gham (grief), udaas (melancholy). Absorbed silently to '
               'protect family harmony. Anger (gussa/krodh): Gussa suppressed toward elders, '
               'acceptable toward subordinates. Krodh is intense destructive anger. Izzat '
               'violation is primary elicitor. Somaticised through reddening face and gnashing of '
               'teeth. Conceptualised as fluid, storm, or wild animal. Fear (darr/ghabrahat): '
               'Direct acknowledgement rare. Triggers: izzat violation and social judgement. '
               'Burden expressed through dil par patthar rakhna - placing a stone on the heart. '
               'Surprise (ashcharya): Less exuberant than English. Terms: ashcharya, adbhut. '
               'Exclamation arre marks sudden shift. Disgust (ghrina): No direct English '
               'equivalent. Maps to moral violation and ritual impurity rooted in Hindu '
               'purity-pollution norms, not sensory repulsion.'},
 'rus': {'p1': 'Russian emotional expression is heavily characterized by a cultural norm of '
               'stoicism in public spaces. Russians are culturally discouraged from displaying '
               'strong positive or negative emotions publicly, and actions like smiling at '
               'strangers are often considered insincere or inappropriate. In stark contrast to '
               'this public restraint, there is a deep cultural allowance for intense, uninhibited '
               'emotional expression within private, trusted relationships. The culture places '
               'significant value on enduring suffering, endurance, and melancholy, viewing these '
               'states as possessing an almost virtuous quality. This is encapsulated in the '
               'untranslatable concept of toska, which describes a profound longing or melancholy. '
               'While daily verbal expression may be restrained, the Russian literary and poetic '
               'tradition serves as a vibrant channel for emotions that are otherwise suppressed. '
               'Consequently, many complex emotional concepts exist primarily within this elevated '
               'literary register rather than in everyday speech. Furthermore, emotions are '
               'closely tied to physical embodiment, as linguistic collocations heavily treat the '
               'body as an organ of emotional expression. However, it is important to note that '
               'online emotional expression on platforms like VKontakte or Telegram has developed '
               'distinct, more disinhibited norms that diverge from traditional offline stoicism.',
         'p2': 'Joy is known as radost, but it is typically expressed privately or exclusively '
               'within close, trusted relationships. Communal celebration is reserved for specific '
               'occasions, as there is a widespread cultural suspicion of excessive or unearned '
               "public positivity. Sadness is conceptualised as grust' or the deeper toska, which "
               'represents a profound melancholy lacking a direct object. Enduring this sadness '
               'quietly is highly valued, reflecting the cultural virtue of suffering and '
               'resilience. Anger, termed gnev, is subject to strict public suppression but can '
               'reach extreme intensity in private settings. It becomes acceptable to display '
               'publicly only under specific conditions where the grievance is widely recognised '
               'as justified. Fear is termed strakh, and it is often expressed indirectly through '
               'dark humour and deflection rather than through direct, vulnerable acknowledgment. '
               "It may also be channelled through physical descriptions of the body's reaction "
               'rather than naming the emotion itself. Surprise is generally less effusive than '
               'its English equivalents. The linguistic markers used to indicate shock or surprise '
               'convey different levels of intensity and are applied more conservatively. Disgust '
               'is frequently framed in moral and aesthetic terms rather than purely physical '
               'revulsion. It is often expressed through an elevated literary or ironic register '
               'rather than via blunt, direct statements.'},
 'hau': {'p1': 'Hausa emotional expression is heavily shaped by Islamic cultural influence and the '
               'core concept of kunya, which dictates a profound sense of shame and reserve in '
               'social interactions. This cultural framework requires emotional restraint, '
               'particularly in the presence of elders or those owed respect, ensuring that '
               'communal harmony is prioritized over individual outbursts. Another guiding '
               'cultural value is haƙuri, which prescribes patience, composure, and acceptance '
               'under distress or hardship. Consequently, Hausa speakers express emotions through '
               'a rich system of conventionalised indirect patterns rather than direct verbal '
               'statements. These indirect channels include deliberate silence, specific hand '
               'gestures, paralinguistic sounds, and the strategic use of proverbs to convey '
               'feelings without violating social norms. Furthermore, emotional language '
               'frequently relies on bodily metaphors, particularly using the word ciki, meaning '
               'stomach or heart area, to locate feeling within the physical body. Emotional '
               'display is also strictly moderated by gender, as what is considered acceptable '
               'expression differs significantly for men and women within the patriarchal '
               'structure. In addition, the deeply institutionalised belief in aljannu, or '
               'spirits, and tsafi, or witchcraft, functions as a powerful tool for social control '
               'and emotional regulation. Women, who face restrictive patriarchal structures, '
               'sometimes channel suppressed anger through indigenous prose fiction and '
               'metaphorical bodily expressions to safely voice dissent.',
         'p2': 'Joy is articulated as farin ciki, which translates literally to white stomach, '
               'framing happiness as a positive bodily state. Rather than through individual '
               'displays of pleasure, joy is expressed via communal celebrations, incorporating '
               'traditional songs and proverbs. Sadness is conceptualised as baƙin ciki, '
               'translating to black stomach. When dealing with grief, condolence greetings known '
               "as gaisuwar ta'aziya rely heavily on Islamic prayers and euphemisms, strictly "
               'avoiding direct verbal statements about death. Anger features multiple verbs of '
               'varying intensity, such as fusata, tunzura, harzua, and hasala, but direct '
               'expression is suppressed by the rules of kunya. Instead, anger is signalled '
               'through visual bodily cues like zare ido, meaning to pull the eye, or contempt '
               'idioms like sha kunu, meaning to drink gruel. Fear is strongly tied to the '
               'supernatural, particularly anxiety regarding aljannu and tsafi. The pervasive fear '
               'of spirit possession operates as a mechanism of social control, meaning fear is '
               'often expressed through religious or superstitious frameworks. Surprise is '
               'communicated through exclamatory sounds and facial mimicry rather than explicit '
               'verbal statements. This non-verbal approach remains consistent with the broader '
               'cultural norm of indirect emotional expression. Disgust is framed primarily as a '
               'moral and religious violation rather than a physical or sensory repulsion. Taboo '
               'violations that trigger disgust are sanctioned through social stigma and the '
               'threat of supernatural consequences.'},
 'kin': {'p1': 'The emotional landscape in Rwanda is fundamentally shaped by the post-genocide '
               'cultural context, where emotional restraint and silence are deeply sanctioned '
               'responses to trauma and distress. Rwanda possesses a documented culture of '
               'silence, meaning that strong emotions are expected to be processed internally '
               'rather than expressed in public spaces. Emotional management is largely governed '
               'by the concept of agaciro, which translates to dignity and self-worth, placing a '
               'premium on composure. Consequently, collective identity is prioritized over '
               'individual emotional display, making community solidarity the primary vessel for '
               'feelings. Communal rituals and structured environments, such as the gacaca justice '
               'proceedings, provide the sanctioned contexts for collective emotional expression. '
               'Vernacular memory practices and community solidarity create culturally specific '
               'avenues for mourning and processing shared history. Emotions tied directly to the '
               'collective trauma of the genocide do not map cleanly onto Western diagnostic or '
               'emotional categories. Thus, Rwandan emotional expression operates as a highly '
               'regulated system of communal processing and dignified restraint rather than a lack '
               'of feeling.',
         'p2': 'Joy is expressed through communal celebration framed heavily by the concept of '
               'agaciro and collective achievement. Individual displays of joy are expected to be '
               'modest, whereas communal joys during national events, church services, or family '
               'milestones are much more visible and acceptable. Sadness is normatively held '
               'internally as part of the broader culture of silence and dignified restraint. '
               'Public mourning is always communal and highly ritualised, relying on vernacular '
               'mourning practices through community memory rather than direct, individual verbal '
               'grief. Anger is publicly suppressed because overt individual aggression violates '
               'the sanctioned culture of silence and restraint. Instead, collective anger is '
               'processed formally through community mechanisms like the gacaca proceedings, '
               'maintaining a communal rather than an individual frame. Fear is prominently '
               'encapsulated by the term ihahamuka, a Rwanda-specific panic and fear response '
               'rooted in genocide trauma that lacks a Western equivalent. This concept combines '
               'feelings of fear, profound bodily distress, and collective memory into a unified '
               'expression. Surprise relies on restrained interpersonal cues rather than overt '
               'verbal exclamations. Exclamative structures using the question marker mbêga and '
               'manner noun ukūntu are characteristic markers, and the interjection yō signals '
               'sudden amazement. Disgust is expressed through social and moral framing tied to '
               'community reputation. Ishyano, meaning ritual impurity, and kuneena, the '
               'institutionalised avoidance of the morally contaminating, are primary mechanisms. '
               'The verb vugisha encodes the act of disgusting or sickening others.'},
 'sun': {'p1': 'Sundanese emotional expression is deeply rooted in the Austronesian cultural '
               'philosophy of Silih Asah, Silih Asih, and Silih Asuh, meaning mutual learning, '
               'mutual affection, and mutual care. This philosophical framework structures '
               'emotional expression as a fundamentally relational and communal experience rather '
               'than a purely individual one. Additionally, Islamic religious principles strongly '
               'influence emotional norms, demanding significant restraint, particularly regarding '
               'negative emotions like anger. In most interpersonal interactions, maintaining a '
               'flat facial expression serves as the dominant social cue to preserve harmony. '
               'Because overt facial displays are restricted, high vocal intonation and physical '
               'pointing gestures replace direct verbal emotional expression. Sundanese speakers '
               'also frequently utilize local cooperative frameworks like gotong royong to channel '
               'feelings into collective action. Pamali, the concept of taboo, and Islamic norms '
               'jointly regulate public emotional expression. The Lemes speech register cushions '
               'emotionally threatening communication and softens the expression of negative '
               'feelings. Consequently, research indicates that Sundanese speakers exhibit higher '
               'empathy orientations compared to other Indonesian ethnic groups.',
         'p2': 'Joy is expressed through communal celebrations and shared activities like the '
               'botram tradition that actively reflect the philosophy of Silih Asih. Individual '
               'joy is consistently framed in terms of relational harmony and community benefit '
               'rather than isolated personal pleasure. The concept of bodas, meaning white, '
               'encodes purity and happiness, and happiness is defined as virtuous living and '
               'inner calm rather than hedonic pleasure. Sadness is termed sedih or galau, and it '
               'is expressed indirectly through highly restrained body language and specific vocal '
               'intonations. The masking norm of crying in the heart while smiling on the face is '
               'culturally embedded. Grief is also channelled through traditional music, where '
               'madenda tuning evokes a sense of sacred melancholy. Anger is primarily non-verbal '
               'because direct verbal anger is socially discouraged and considered disruptive to '
               'relational harmony. Instead, individuals turn to Islamic coping responses such as '
               'wudhu for ritual washing, istighfar for seeking forgiveness, and the deliberate '
               'practice of patience. Fear is captured by the blended concepts of kawatir and '
               'takut, which merge anxiety and fear into a unified emotional state. Supernatural '
               'and spiritual concerns, heavily influenced by Islamic beliefs, remain prominent '
               'triggers for these fearful expressions. Surprise is uniquely marked by the '
               'exclamatory word meuni, meaning such or how, which frequently collocates with '
               'adjectives and interjections. The intensifier pisan amplifies affect, and the '
               'interjection Duh anchors climactic emotional moments. Disgust is expressed '
               'primarily through moral and religious purity framing, with najis, meaning ritually '
               'unclean, functioning as the primary trigger for jijik, the Sundanese term for '
               'disgust.'},
 'yor': {'p1': 'Yoruba emotional expression is defined by a collectivist community orientation '
               'where feelings are rarely stated directly. Instead, emotions are channelled '
               'through an elaborate system of proverbs known as òwe, as well as through '
               'metaphors, folksongs, and oral poetry. Traditional festivals and communal '
               'gatherings provide the culturally sanctioned spaces for the collective processing '
               'of immense joy and grief. Outside of these communal rituals, individual emotional '
               'display is strictly moderated by powerful social norms regarding face, age, and '
               'respect. Direct confrontation is considered culturally inappropriate, meaning '
               'negative emotions are particularly subject to proverbial indirection. Proverbs '
               'function dynamically as both weapons of social power and diplomatic tools for '
               'conflict resolution. Furthermore, meaning is heavily supplemented by non-verbal '
               'semiotics, including specific hand gestures and facial expressions that carry '
               'exact cultural weight. Therefore, Yoruba emotional communication relies on a '
               'shared, highly contextual understanding of oral literature and bodily metaphor '
               'rather than explicit individual declaration.',
         'p2': 'Joy is expressed collectively through folksongs and traditional festivals that '
               'function as communal soul-menders. It is consistently framed in terms of community '
               'wellbeing, family harmony, and shared prosperity rather than isolated personal '
               'pleasure. Sadness is articulated indirectly through the recitation of proverbs, '
               'traditional dirges, and folksongs. The culture employs a strong somatic '
               'orientation, frequently using bodily organs like the heart metaphorically to '
               'locate and process sorrow. Anger relies heavily on proverbs acting as '
               'face-threatening acts, where powerful sayings function as indirect threats or '
               'insults to avoid discouraged direct confrontation. When described physically, '
               'anger utilizes heat and fire metaphors to illustrate how the emotion violently '
               'overwhelms the body. Fear is powerfully signalled non-verbally through the face of '
               'earnest, a culturally specific facial expression indicating grave seriousness that '
               'is easily misread cross-culturally. Additionally, speakers rely on describing '
               'sudden bodily sensations to locate fear rather than stating the emotion '
               'abstractly. Surprise is communicated via exclamatory interjections and abrupt '
               'shifts in body language. The symbolic placement of an ààlè, using objects like '
               'sand, leaves, or red cloth, conveys non-verbal warning and shock. Disgust is '
               'expressed through proverbs that explicitly invoke moral violation and the breach '
               'of social taboos. It represents a collective community judgement regarding '
               'disrespect for cultural norms rather than an individual sensory reaction.'},
 'vmw': {'p1': 'Emakhuwa emotional expression is profoundly shaped by its matrilineal Bantu social '
               'structure, which dictates distinct gender roles for processing feelings. Women '
               'hold a central role in communal ritual and the public expression of grief, whereas '
               'male gender norms strictly suppress vulnerable emotions like fear and sadness. '
               'Displaying grief loudly and publicly through ritualised mourning is viewed as a '
               'moral obligation to the community rather than a private, individual act. '
               'Conversely, collective joy is channelled through highly structured avenues like '
               'the tufo competitive dance and the Nakhula ancestral dance. Emotional management '
               'is also governed by the concept of ehaya, representing a form of shame tied '
               'intrinsically to communal reputation rather than individual guilt. Furthermore, '
               'collective fear and anger are frequently articulated through culturally specific '
               'idioms of sorcery that possess no direct Western equivalent. It is important to '
               'note that very little academic literature exists regarding Emakhuwa emotional '
               'expression in online contexts, so these offline anthropological norms remain the '
               'primary point of reference.',
         'p2': 'Joy is expressed collectively through events like the tufo competitive dance and '
               'Nakhula ancestral dance during harvests and marriages. Happiness is conceptualised '
               'as a collective communal experience tied to shared celebration and ancestral '
               'ritual. Sadness is expressed through ritualised, public wailing at funerals, which '
               'operates as a strict moral obligation. This loud, collective expression replaces '
               'quiet private grief, while indirect sadness is also processed through traditional '
               'dance and oral tradition. Anger is frequently expressed through sorcery idioms, '
               'utilizing terms like havara for leopard or sorcerer, and ekuluwe for pig to '
               'describe household discord. In broader political contexts, collective resistance '
               'and anger are signalled through phrases like anamalala, meaning it is over. Fear '
               'is heavily tied to sorcery and the mgosyo taboo system regarding hot and cold '
               'states. Anxiety concerning sorcerers among neighbours serves as the dominant '
               'expression of fear, while mgosyo transgressions produce a unique fear-guilt blend '
               'with physical bodily consequences. Surprise possesses very limited direct lexical '
               'expression in the language. Instead, sudden shock is conveyed through specific '
               'interjections and exaggerated body language rather than explicit vocabulary. '
               'Disgust is primarily a moral emotion associated with violations of mgosyo taboos '
               'and subsequent sorcery accusations. Physical or sensory disgust, as it is '
               'understood within Western frameworks, is significantly less prominent.'},
 'pcm': {'p1': 'Nigerian Pidgin, often referred to as Naijà, functions as a vital cross-ethnic '
               "lingua franca that bridges Nigeria's incredibly diverse cultural groups. Because "
               'it deliberately spans various communities, its emotional expression tends to be '
               'much more direct than indigenous languages like Yoruba or Hausa, while still '
               'maintaining its own distinct culturally Nigerian character. Corpus research '
               'indicates that negative sentiment is notably more prevalent in Nigerian language '
               'communities compared to other African language groups. A defining characteristic '
               'of the language is its heavy reliance on unique interjections, which serve as the '
               'most prominent and distinct emotional markers. Furthermore, emotional intensity is '
               'frequently conveyed through the grammatical process of reduplication, where '
               'repeating a word amplifies its feeling. Consistent with wider West African '
               'linguistic patterns, Nigerian Pidgin utilizes bodily sensation constructions that '
               'place emotion directly in the physical body rather than naming it abstractly. '
               'Online, this directness is further amplified, heavily mixing these culturally '
               'specific interjections with global social media norms and emojis.',
         'p2': 'Joy is frequently marked by the exclamatory interjection omo, which signals '
               'intense excitement, admiration, or positive shock. The language relies on communal '
               'celebratory phrasing, strongly favouring these distinctive Pidgin interjections '
               'over their standard English equivalents to express happiness. Sadness is expressed '
               'through bodily sensation constructions, most notably the phrase e pain me, which '
               'locates the sorrow as physical pain. Resigned sorrow or disbelief is captured by '
               'the marker na wa o, while online expressions frequently pair English interjections '
               'alongside sadness emojis. Anger is expressed very directly through phrases like I '
               'vex, meaning I am angry. Intensity is added through reduplication, such as saying '
               'vex vex, and anger frequently co-occurs with expressions of disgust within the '
               'exact same utterance. Fear is conveyed through phrases like I fear am, meaning I '
               'am afraid of it, grounding the feeling in bodily sensations rather than abstract '
               'names. Sharp, repeated pain or fearful distress is also expressed through '
               'sound-symbolic reduplication, such as the term CHUK CHUK. Surprise is marked by '
               'highly distinctive Pidgin interjections like chai and haba. Interestingly, these '
               'shocked expressions tend to occur much more frequently in contexts of negative '
               'surprise than in positive ones. Disgust is widely expressed using the direct '
               'phrase e no good, meaning it is not good. This relies heavily on a dominant moral '
               'framing and frequently clusters together with anger in the same sentence.'}}


def build_cultural_block(language_code, prompt_key):
    content = CULTURAL_CONTENT.get(language_code, {})
    if prompt_key == "p0":
        return ""
    if prompt_key in ("p1", "p2"):
        return content.get(prompt_key, "").strip()
    if prompt_key == "p3":
        combined = "\n\n".join(
            part for part in [
                content.get("p1", "").strip(),
                content.get("p2", "").strip(),
            ]
            if part
        )
        return combined[:3200]
    raise ValueError(f"Unknown prompt key: {prompt_key}")


INSTRUCTION = (
    "Read the text below and identify which emotions it expresses.\n"
    "Available emotions: joy, sadness, anger, fear, surprise, disgust\n"
    "A text may express multiple emotions, one emotion, or none at all.\n\n"
    "Instructions:\n"
    "- Respond ONLY with a comma-separated list of emotion labels from the list above.\n"
    "- Use lowercase exactly as written above.\n"
    "- If no emotion is present, respond with the single word: neutral\n"
    "- Do not add any explanation, punctuation, or extra text."
)


def select_few_shot_examples(target_code, config_key, k=FEW_SHOT_K):
    pool = POOL_FUNCTIONS[config_key](target_code)
    frames = [
        SOURCE_TRAIN_DATA[code]
        for code in pool
        if code in SOURCE_TRAIN_DATA
    ]
    if not frames:
        raise ValueError(
            f"{target_code}/{config_key}: no available source-language training data"
        )

    pool_frame = pd.concat(frames, ignore_index=True)
    examples_by_emotion = {emotion: [] for emotion in EMOTION_ORDER}

    for _, row in pool_frame.iterrows():
        text = str(row.get("text", "")).strip()
        if not text:
            continue
        labels = [
            emotion for emotion in EMOTION_ORDER
            if pd.notna(row.get(emotion, 0)) and int(row.get(emotion, 0)) == 1
        ]
        if not labels:
            continue
        for emotion in labels:
            examples_by_emotion[emotion].append((text, labels))


    rng = random.Random(RANDOM_SEED + ALL_TARGET_CODES.index(target_code))
    lines = [
        "Here are some example texts in related languages and their emotions:\n"
    ]
    example_index = 1

    for emotion in EMOTION_ORDER:
        candidates = examples_by_emotion[emotion]
        if not candidates:
            continue
        single_label = [
            item for item in candidates if len(item[1]) == 1
        ]
        candidate_pool = single_label if single_label else candidates
        selected = rng.sample(candidate_pool, min(k, len(candidate_pool)))

        for text, labels in selected:
            shortened = text[:120] + "..." if len(text) > 120 else text
            lines.append(
                f'  Example {example_index}: "{shortened}" -> '
                + ", ".join(labels)
            )
            example_index += 1

    if example_index == 1:
        raise ValueError(f"{target_code}/{config_key}: no labelled examples selected")
    lines.append("")
    return "\n".join(lines) + "\n"


## 6. Freeze validation selections


In [ ]:
p2_path, p3_path = (
    f"{P2_DIR}/phase2_validation_raw_results_gemma4.json",
    f"{P3_DIR}/phase3_xlmr_validation_scores.json"
)

if not all(os.path.exists(p) for p in (p2_path, p3_path)):
    raise FileNotFoundError("Run Phase 2 and Phase 3 validation before Phase 4.")

p2, p3 = [json.load(open(p, encoding="utf-8")) for p in (p2_path, p3_path)]

# Phase 3 JSON stores keys as track_a_C1 -- helper handles both formats
# and falls back from macro_f1_raw to macro_f1 when raw is absent
def _p3_entry(p3_lang, cfg):
    return p3_lang.get(f"track_a_{cfg}") or p3_lang.get(cfg)

def _p3_score(p3_lang, cfg):
    entry = _p3_entry(p3_lang, cfg)
    if not isinstance(entry, dict):
        return None
    return entry.get("macro_f1_raw") or entry.get("macro_f1")

selection = {
    "selection_split": "validation",
    "prompt_source": p2_path,
    "config_source": p3_path,
    "languages": {}
}

rows = []
for lang in RUN_LANGS:
    if not all(p in p2.get(lang, {}) for p in PROMPT_ORDER):
        raise ValueError(f"Incomplete Phase 2 validation results for {lang}")

    configs = [c for c in CONFIG_ORDER if _p3_score(p3.get(lang, {}), c) is not None]
    if not configs:
        raise ValueError(f"No Phase 3 validation result for {lang}")

    bp = max(PROMPT_ORDER, key=lambda p: p2[lang][p]["macro_f1"])
    bc = max(configs, key=lambda c: _p3_score(p3[lang], c))

    selection["languages"][lang] = {
        "best_prompt": bp,
        "best_config": bc,
        "phase2_validation_f1": {p: p2[lang][p]["macro_f1"] for p in PROMPT_ORDER},
        "phase3_validation_f1": {c: _p3_score(p3[lang], c) for c in configs}
    }

    rows.append({
        "Language code":                                  lang.upper(),
        "Validation-selected prompt":                     bp,
        "Validation-selected transfer configuration":     bc,
        "Selected prompt validation macro-F1":            p2[lang][bp]["macro_f1"],
        "Selected configuration validation macro-F1":     _p3_score(p3[lang], bc),
        "Published Track A test best macro-F1 (context only)": BENCHMARK_A[lang],
        "Published Track C test best macro-F1 (context only)": BENCHMARK_C[lang],
        "Benchmark comparison status":                    BENCHMARK_STATUS,
    })

FEWSHOT_BLOCKS = {
    (l, v["best_config"]): select_few_shot_examples(l, v["best_config"])
    for l, v in selection["languages"].items()
}

atomic_json_write(SELECTION_PATH, selection)
pd.DataFrame(rows).to_csv(f"{P4_DIR}/phase4_validation_selections.csv", index=False)

atomic_json_write(f"{P4_DIR}/phase4_selected_fewshot_blocks.json", {
    l: {
        "selected_config":    v["best_config"],
        "source_pool":        POOL_FUNCTIONS[v["best_config"]](l),
        "fewshot_k_per_emotion": FEW_SHOT_K,
        "rendered_fewshot_block": FEWSHOT_BLOCKS[(l, v["best_config"])]
    }
    for l, v in selection["languages"].items()
})

print("Validation selections frozen before loading target test data.")
display(pd.DataFrame(rows).drop(columns=["Benchmark comparison status"], errors="ignore"))

Validation selections frozen before loading target test data.


,Language code,Validation-selected prompt,Validation-selected transfer configuration,Selected prompt validation macro-F1,Selected configuration validation macro-F1,Published Track A test best macro-F1 (context only),Published Track C test best macro-F1 (context only)
0,ENG,p0,C3,0.6274,0.7634,0.823,0.797
1,HIN,p3,C1,0.7752,0.8433,0.926,0.919
2,RUS,p1,C3,0.8414,0.8784,0.901,0.906
3,HAU,p2,C1,0.6063,0.6792,0.751,0.709
4,KIN,p1,C3,0.4550,0.4048,0.657,0.519
5,SUN,p0,C1,0.6516,0.5035,0.550,0.467
6,YOR,p1,C2,0.3684,0.2888,0.461,0.359
7,VMW,p3,C2,0.1581,0.2038,0.325,0.210
8,PCM,p1,C2,0.5639,0.5708,0.674,0.674


## 7. Load target test data


In [ ]:
import shutil, os

LOCAL_CACHE = "/content/parquet_cache"
os.makedirs(LOCAL_CACHE, exist_ok=True)

# Copy all parquets from Drive cache to local disk
copied = 0
for fname in os.listdir(CACHE_DIR):
    if fname.endswith(".parquet"):
        src = f"{CACHE_DIR}/{fname}"
        dst = f"{LOCAL_CACHE}/{fname}"
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
print(f"Copied {copied} parquet files to local disk.")

Copied 0 parquet files to local disk.


In [ ]:
_ORIGINAL_CACHE_DIR = CACHE_DIR

def load_split(language, split):
    local_path = f"{LOCAL_CACHE}/{language}_{split}.parquet"
    drive_path = f"{_ORIGINAL_CACHE_DIR}/{language}_{split}.parquet"

    if os.path.exists(local_path):
        frame = pd.read_parquet(local_path)
    elif os.path.exists(drive_path):
        shutil.copy2(drive_path, local_path)
        frame = pd.read_parquet(local_path)
    else:
        dataset = load_dataset("brighter-dataset/BRIGHTER-emotion-categories", language)
        frame = dataset[split].to_pandas()
        frame.to_parquet(local_path, index=False)
        frame.to_parquet(drive_path, index=False)

    for emotion in EMOTION_ORDER:
        if emotion not in frame.columns:
            frame[emotion] = 0
    if "text" not in frame.columns:
        raise ValueError(f"{language}/{split}: text column is missing")
    return frame.reset_index(drop=True)

print("load_split now reads from local disk.")

load_split now reads from local disk.


In [ ]:
frozen=json.load(open(SELECTION_PATH,encoding="utf-8"))
assert frozen["selection_split"]=="validation" and set(frozen["languages"])==set(RUN_LANGS)
TEST_DATA={lang:load_split(lang,"test") for lang in RUN_LANGS}
for lang,frame in TEST_DATA.items(): print(f"{lang}: test rows={len(frame)}")
print("Target test data loaded only after selection was frozen.")


eng: test rows=5528
hin: test rows=2020
rus: test rows=2000
hau: test rows=2160
kin: test rows=2462
sun: test rows=1852
yor: test rows=3000
vmw: test rows=1554
pcm: test rows=3740
Target test data loaded only after selection was frozen.


## 8. Gemma inference


In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForMultimodalLM, AutoProcessor

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN)

gemma_processor = gemma_model = inner_tokenizer = None
if RUN_TEST_INFERENCE:
    print(f"Loading {GEMMA_MODEL}")
    gemma_processor = AutoProcessor.from_pretrained(
        GEMMA_MODEL, token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE,
        trust_remote_code=True
    )
    gemma_model = AutoModelForMultimodalLM.from_pretrained(
        GEMMA_MODEL, token=HF_TOKEN, cache_dir=LOCAL_MODEL_CACHE,
        trust_remote_code=True, dtype="auto", device_map="auto",
        attn_implementation="sdpa"
    ).eval()
    inner_tokenizer = getattr(gemma_processor, "tokenizer", gemma_processor)
    inner_tokenizer.padding_side = "left"
    if inner_tokenizer.pad_token_id is None:
        inner_tokenizer.pad_token_id = inner_tokenizer.eos_token_id
    GENERATION_CONFIG["pad_token_id"] = inner_tokenizer.pad_token_id
else:
    print("Inference disabled: existing predictions will be reused.")

def apply_chat_template(messages):
    options = {
        "tokenize": False,
        "add_generation_prompt": True,
        "enable_thinking": False,
    }
    try:
        return gemma_processor.apply_chat_template(messages, **options)
    except TypeError:
        options.pop("enable_thinking")
        return gemma_processor.apply_chat_template(messages, **options)


def build_prompt(language_code, text, prompt_key="p0", config_key=None):
    cultural = build_cultural_block(language_code, prompt_key)
    if cultural:
        system_content = (
            "You are a culturally-aware emotion classification system.\n\n"
            + cultural
            + "\n\n"
            + INSTRUCTION
        )
    else:
        system_content = (
            "You are an emotion classification system.\n"
            + INSTRUCTION
        )

    if config_key is not None:
        system_content += (
            "\n\n"
            + FEWSHOT_BLOCKS[(language_code, config_key)].rstrip()
        )

    messages = [
        {
            "role": "user",
            "content": f"{system_content}\n\nText: {text}",
        }
    ]
    return apply_chat_template(messages) + "Emotions:"


def parse_generated_labels(raw_text):
    raw_text = raw_text.strip().lower()
    if raw_text in ("", "neutral"):
        return []
    return [
        label.strip()
        for label in raw_text.split(",")
        if label.strip() in EMOTION_ORDER
    ]


def prediction_path(output_directory, split, language, condition):
    return f"{output_directory}/{split}_{language}_{condition}.json"


def run_or_load_condition(
    frame,
    split,
    language,
    condition,
    prompt_key,
    config_key,
    output_directory,
    run_inference,
):
    output_path = prediction_path(
        output_directory,
        split,
        language,
        condition,
    )
    expected_metadata = {
        "split": split,
        "language": language,
        "condition": condition,
        "prompt": prompt_key,
        "config": config_key,
        "emotions": EMOTION_ORDER,
    }

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as handle:
            saved = json.load(handle)
        metadata_matches = all(
            saved.get(key) == value
            for key, value in expected_metadata.items()
        )
        if metadata_matches:
            y_true = np.asarray(saved["y_true"], dtype=int)
            y_pred = np.asarray(saved["y_pred"], dtype=int)
            expected_y_true = labels_to_matrix(frame)
            if np.array_equal(y_true, expected_y_true):
                print(f"Reusing {os.path.basename(output_path)}")
                return score_predictions(y_true, y_pred, language)

    if not run_inference:
        raise FileNotFoundError(
            f"Current prediction not found and inference is disabled: {output_path}"
        )

    y_true = labels_to_matrix(frame)
    y_pred = np.zeros_like(y_true)
    texts = frame["text"].astype(str).tolist()
    progress_path = f"{output_path}.progress.json"
    start_index = 0

    if os.path.exists(progress_path):
        with open(progress_path, "r", encoding="utf-8") as handle:
            progress = json.load(handle)
        if all(
            progress.get(key) == value
            for key, value in expected_metadata.items()
        ):
            stored_predictions = np.asarray(progress["y_pred"], dtype=int)
            if stored_predictions.shape == y_pred.shape:
                y_pred = stored_predictions
                start_index = int(progress["processed"])
                print(
                    f"Resuming {language}/{condition} "
                    f"from {start_index}/{len(texts)}"
                )

    inner_tokenizer.padding_side = "left"
    for batch_start in range(start_index, len(texts), BATCH_SIZE):
        batch_texts = texts[batch_start: batch_start + BATCH_SIZE]
        prompts = [
            build_prompt(
                language,
                text,
                prompt_key=prompt_key,
                config_key=config_key,
            )
            for text in batch_texts
        ]
        inputs = gemma_processor(
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(gemma_model.device)

        with torch.no_grad():
            output_ids = gemma_model.generate(
                **inputs,
                **GENERATION_CONFIG,
            )

        prompt_length = inputs["input_ids"].shape[1]
        for local_index, output in enumerate(output_ids):
            generated_ids = output[prompt_length:]
            raw = gemma_processor.decode(
                generated_ids,
                skip_special_tokens=True,
            )
            detected = parse_generated_labels(raw)
            row_index = batch_start + local_index
            for emotion_index, emotion in enumerate(EMOTION_ORDER):
                y_pred[row_index, emotion_index] = int(emotion in detected)

        processed = min(batch_start + BATCH_SIZE, len(texts))
        progress_payload = {
            **expected_metadata,
            "processed": processed,
            "y_pred": y_pred.tolist(),
        }
        atomic_json_write(progress_path, progress_payload)
        if processed % 50 == 0 or processed == len(texts):
            print(
                f"{language}/{condition}: "
                f"{processed}/{len(texts)} samples"
            )

    final_payload = {
        **expected_metadata,
        "y_true": y_true.tolist(),
        "y_pred": y_pred.tolist(),
    }
    atomic_json_write(output_path, final_payload)
    if os.path.exists(progress_path):
        os.remove(progress_path)
    return score_predictions(y_true, y_pred, language)


if gemma_model is not None and torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Gemma inference engine ready. VRAM: {allocated:.1f}/{total:.1f} GB")
elif gemma_model is not None:
    print("Gemma inference engine ready on CPU.")


Loading unsloth/gemma-4-31B-it-unsloth-bnb-4bit


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Gemma inference engine ready. VRAM: 18.6/42.4 GB


## 9. Four-condition test ablation


In [ ]:
RESULTS_PATH=f"{P4_DIR}/phase4_test_ablation_results.json"
results=json.load(open(RESULTS_PATH,encoding="utf-8")) if os.path.exists(RESULTS_PATH) else {}
rows=[]
for lang in RUN_LANGS:
    chosen=frozen["languages"][lang]; bp,bc=chosen["best_prompt"],chosen["best_config"]
    specs={"neither":("p0",None),"prompt_only":(bp,None),
           "config_only":("p0",bc),"both":(bp,bc)}
    results.setdefault(lang,{}); seen={}
    print(f"\nTest ablation: {LANGUAGES[lang]['name']} (prompt={bp.upper()}, config={bc})")
    for condition,(prompt,config) in specs.items():
        key=(prompt,config)
        if key in seen:
            src=load_json(prediction_path(TEST_PRED_DIR,"test",lang,seen[key]))
            src.update(condition=condition,prompt=prompt,config=config)
            atomic_json_write(prediction_path(TEST_PRED_DIR,"test",lang,condition),src)
            score=score_predictions(src["y_true"],src["y_pred"],lang)
        else:
            score=run_or_load_condition(TEST_DATA[lang],"test",lang,condition,prompt,config,
                                        TEST_PRED_DIR,RUN_TEST_INFERENCE)
            seen[key]=condition
        results[lang][condition]=score; atomic_json_write(RESULTS_PATH,results)
        rows.append({
            "Language code":                                  lang.upper(),
            "Ablation condition":                             condition,
            "Validation-selected prompt":                     bp,
            "Validation-selected transfer configuration":     bc,
            "Macro-F1":                                       score["macro_f1"],
            "Published Track A test best macro-F1 (context only)": BENCHMARK_A[lang],
            "Published Track C test best macro-F1 (context only)": BENCHMARK_C[lang],
            "Benchmark comparison status":                    BENCHMARK_STATUS,
        })
        print(f"  {condition:12s}: {score['macro_f1']:.4f}")
pd.DataFrame(rows).to_csv(f"{P4_DIR}/phase4_test_ablation_results.csv",index=False)



Test ablation: English (prompt=P0, config=C3)
Reusing test_eng_neither.json
  neither     : 0.6245
  prompt_only : 0.6245
Reusing test_eng_config_only.json
  config_only : 0.6290
  both        : 0.6290

Test ablation: Hindi (prompt=P3, config=C1)
Reusing test_hin_neither.json
  neither     : 0.8043
Reusing test_hin_prompt_only.json
  prompt_only : 0.8393
Reusing test_hin_config_only.json
  config_only : 0.8384
Reusing test_hin_both.json
  both        : 0.8496

Test ablation: Russian (prompt=P1, config=C3)
Reusing test_rus_neither.json
  neither     : 0.8071
Reusing test_rus_prompt_only.json
  prompt_only : 0.8371
Reusing test_rus_config_only.json
  config_only : 0.8504
Reusing test_rus_both.json
  both        : 0.8559

Test ablation: Hausa (prompt=P2, config=C1)
Reusing test_hau_neither.json
  neither     : 0.5491
Reusing test_hau_prompt_only.json
  prompt_only : 0.5790
Reusing test_hau_config_only.json
  config_only : 0.5317
Reusing test_hau_both.json
  both        : 0.5741

Test abl

## 10. Bootstrap inference


In [ ]:
_pw_path  = f"{P4_DIR}/phase4_pairwise_bootstrap.csv"
_int_path = f"{P4_DIR}/phase4_interaction_bootstrap.csv"

if os.path.exists(_pw_path) and os.path.exists(_int_path):
    pairwise_table   = pd.read_csv(_pw_path)
    interaction_table = pd.read_csv(_int_path)
    print("✓ Loaded cached bootstrap results (skipping recomputation)")
else:
    def load_pilot_prediction(split, language, condition):
        directory = (
            VALIDATION_PRED_DIR
            if split == "validation"
            else TEST_PRED_DIR
        )
        path = prediction_path(directory, split, language, condition)
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if payload.get("emotions") != EMOTION_ORDER:
            raise ValueError(f"{path}: unexpected emotion order")
        return (
            np.asarray(payload["y_true"], dtype=int),
            np.asarray(payload["y_pred"], dtype=int),
        )

    def macro_f1_array(y_true, y_pred, language):
        keep = [
            index for index, emotion in enumerate(EMOTION_ORDER)
            if emotion in active_emotions(language)
        ]
        return float(
            f1_score(
                y_true[:, keep],
                y_pred[:, keep],
                average="macro",
                zero_division=0,
            )
        )

    def paired_bootstrap(
        y_true,
        prediction_a,
        prediction_b,
        language,
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    ):
        rng = np.random.default_rng(seed)
        sample_count = len(y_true)
        observed = (
            macro_f1_array(y_true, prediction_a, language)
            - macro_f1_array(y_true, prediction_b, language)
        )
        draws = np.empty(n_boot)
        for index in range(n_boot):
            sampled = rng.integers(0, sample_count, sample_count)
            draws[index] = (
                macro_f1_array(y_true[sampled], prediction_a[sampled], language)
                - macro_f1_array(y_true[sampled], prediction_b[sampled], language)
            )
        low, high = np.percentile(draws, [2.5, 97.5])
        p_value = min(
            1.0,
            2 * min(
                float(np.mean(draws <= 0)),
                float(np.mean(draws >= 0)),
            ),
        )
        return observed, low, high, p_value

    def interaction_bootstrap(
        y_true,
        neither,
        prompt_only,
        config_only,
        both,
        language,
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    ):
        rng = np.random.default_rng(seed)
        sample_count = len(y_true)

        def interaction(indices):
            base     = macro_f1_array(y_true[indices], neither[indices],     language)
            prompt   = macro_f1_array(y_true[indices], prompt_only[indices],  language)
            config   = macro_f1_array(y_true[indices], config_only[indices],  language)
            combined = macro_f1_array(y_true[indices], both[indices],         language)
            return combined - prompt - config + base

        full_indices = np.arange(sample_count)
        observed = interaction(full_indices)
        draws = np.empty(n_boot)
        for index in range(n_boot):
            sampled = rng.integers(0, sample_count, sample_count)
            draws[index] = interaction(sampled)

        low, high = np.percentile(draws, [2.5, 97.5])
        p_value = min(
            1.0,
            2 * min(
                float(np.mean(draws <= 0)),
                float(np.mean(draws >= 0)),
            ),
        )
        return observed, low, high, p_value

    PAIRWISE_COMPARISONS = [
        ("prompt_only vs neither", "prompt_only", "neither"),
        ("config_only vs neither", "config_only", "neither"),
        ("both vs neither",        "both",        "neither"),
        ("both vs prompt_only",    "both",        "prompt_only"),
        ("both vs config_only",    "both",        "config_only"),
    ]

    pairwise_rows    = []
    interaction_rows = []

    for language in RUN_LANGS:
        predictions      = {}
        reference_y_true = None

        for condition in ["neither", "prompt_only", "config_only", "both"]:
            y_true, y_pred = load_pilot_prediction("test", language, condition)
            if reference_y_true is None:
                reference_y_true = y_true
            elif not np.array_equal(reference_y_true, y_true):
                raise ValueError(f"{language}: test gold-label mismatch")
            predictions[condition] = y_pred

        for label, condition_a, condition_b in PAIRWISE_COMPARISONS:
            delta, low, high, raw_p = paired_bootstrap(
                reference_y_true,
                predictions[condition_a],
                predictions[condition_b],
                language,
                seed=RANDOM_SEED + RUN_LANGS.index(language),
            )
            pairwise_rows.append({
                "Language":                                       LANGUAGES[language]["name"],
                "Language code":                                  language.upper(),
                "Comparison":                                     label,
                "Macro-F1 difference (first minus second)":       round(delta, 4),
                "95% confidence interval lower bound":            round(low, 4),
                "95% confidence interval upper bound":            round(high, 4),
                "Raw p-value":                                    raw_p,
            })

        effect, low, high, raw_p = interaction_bootstrap(
            reference_y_true,
            predictions["neither"],
            predictions["prompt_only"],
            predictions["config_only"],
            predictions["both"],
            language,
            seed=RANDOM_SEED + RUN_LANGS.index(language),
        )
        interaction_rows.append({
            "Language":                                                       LANGUAGES[language]["name"],
            "Language code":                                                  language.upper(),
            "Interaction effect: both - prompt-only - config-only + neither": round(effect, 4),
            "95% confidence interval lower bound":                            round(low, 4),
            "95% confidence interval upper bound":                            round(high, 4),
            "Raw p-value":                                                    raw_p,
        })

    # Pairwise - one ten-test family
    pairwise_table = pd.DataFrame(pairwise_rows)
    pairwise_reject, pairwise_adjusted, _, _ = multipletests(
        pairwise_table["Raw p-value"].to_numpy(), method="holm",
    )
    pairwise_table["Holm-adjusted p-value"] = pairwise_adjusted
    pairwise_table["Significant after Holm correction (alpha=0.05)"] = pairwise_reject
    pairwise_table["Evaluation split"] = "Test"
    pairwise_table["Statistical test"] = "Paired bootstrap; Holm correction applied"
    pairwise_table["Analysis scope"]   = "Internal Phase 4 CAST ablation"
    _pw_cols = [
        "Language", "Comparison", "Macro-F1 difference (first minus second)",
        "95% confidence interval lower bound", "95% confidence interval upper bound",
        "Raw p-value", "Holm-adjusted p-value",
        "Significant after Holm correction (alpha=0.05)",
        "Evaluation split", "Statistical test", "Analysis scope", "Language code",
    ]
    pairwise_table = pairwise_table[_pw_cols]

    # Interaction - two-test family
    interaction_table = pd.DataFrame(interaction_rows)
    interaction_reject, interaction_adjusted, _, _ = multipletests(
        interaction_table["Raw p-value"].to_numpy(), method="holm",
    )
    interaction_table["Holm-adjusted p-value"] = interaction_adjusted
    interaction_table["Significant after Holm correction (alpha=0.05)"] = interaction_reject
    interaction_table["Interaction interpretation"] = np.where(
        interaction_table["Interaction effect: both - prompt-only - config-only + neither"] < 0,
        "sub-additive",
        np.where(
            interaction_table["Interaction effect: both - prompt-only - config-only + neither"] > 0,
            "synergistic",
            "exactly additive",
        ),
    )
    interaction_table["Evaluation split"] = "Test"
    interaction_table["Statistical test"] = "Paired bootstrap; Holm correction applied"
    interaction_table["Analysis scope"]   = "Internal Phase 4 CAST ablation"
    _int_cols = [
        "Language",
        "Interaction effect: both - prompt-only - config-only + neither",
        "95% confidence interval lower bound", "95% confidence interval upper bound",
        "Raw p-value", "Holm-adjusted p-value",
        "Significant after Holm correction (alpha=0.05)",
        "Interaction interpretation",
        "Evaluation split", "Statistical test", "Analysis scope", "Language code",
    ]
    interaction_table = interaction_table[_int_cols]

    pairwise_table.to_csv(_pw_path,   index=False)
    interaction_table.to_csv(_int_path, index=False)

print("Pairwise bootstrap results")
display(pairwise_table)
print("\nInteraction bootstrap results")
display(interaction_table)

In [ ]:
_pw_path  = f"{P4_DIR}/phase4_pairwise_bootstrap.csv"
_int_path = f"{P4_DIR}/phase4_interaction_bootstrap.csv"

_code_to_name = {k.upper(): v["name"] for k, v in LANGUAGES.items()}

# Pairwise
pw = pd.read_csv(_pw_path)
pw = pw.rename(columns={
    "language":         "Language code",
    "comparison":       "Comparison",
    "delta":            "Macro-F1 difference (first minus second)",
    "ci_low":           "95% confidence interval lower bound",
    "ci_high":          "95% confidence interval upper bound",
    "raw_p":            "Raw p-value",
    "holm_p":           "Holm-adjusted p-value",
    "significant_0.05": "Significant after Holm correction (alpha=0.05)",
})
if "Language code" in pw.columns:
    pw["Language code"] = pw["Language code"].str.upper()
if "Language" not in pw.columns:
    pw.insert(0, "Language", pw["Language code"].map(_code_to_name))
if "Evaluation split" not in pw.columns:
    pw["Evaluation split"] = "Test"
    pw["Statistical test"]  = "Paired bootstrap; Holm correction applied"
    pw["Analysis scope"]    = "Internal Phase 4 CAST ablation"
pw.to_csv(_pw_path, index=False)
print(f"✓ XLM-R pairwise renamed: {list(pw.columns)}")

# Interaction
it = pd.read_csv(_int_path)
it = it.rename(columns={
    "language":         "Language code",
    "interaction":      "Interaction effect: both - prompt-only - config-only + neither",
    "ci_low":           "95% confidence interval lower bound",
    "ci_high":          "95% confidence interval upper bound",
    "raw_p":            "Raw p-value",
    "holm_p":           "Holm-adjusted p-value",
    "significant_0.05": "Significant after Holm correction (alpha=0.05)",
    "interpretation":   "Interaction interpretation",
})
if "Language code" in it.columns:
    it["Language code"] = it["Language code"].str.upper()
if "Language" not in it.columns:
    it.insert(0, "Language", it["Language code"].map(_code_to_name))
if "Evaluation split" not in it.columns:
    it["Evaluation split"] = "Test"
    it["Statistical test"]  = "Paired bootstrap; Holm correction applied"
    it["Analysis scope"]    = "Internal Phase 4 CAST ablation"
it.to_csv(_int_path, index=False)
print(f" XLM-R interaction renamed: {list(it.columns)}")

✓ XLM-R pairwise renamed: ['Language', 'Language code', 'Comparison', 'Macro-F1 difference (first minus second)', '95% confidence interval lower bound', '95% confidence interval upper bound', 'Raw p-value', 'Holm-adjusted p-value', 'Significant after Holm correction (alpha=0.05)', 'Evaluation split', 'Statistical test', 'Analysis scope']
✓ XLM-R interaction renamed: ['Language', 'Language code', 'Interaction effect: both - prompt-only - config-only + neither', '95% confidence interval lower bound', '95% confidence interval upper bound', 'Raw p-value', 'Holm-adjusted p-value', 'Significant after Holm correction (alpha=0.05)', 'Interaction interpretation', 'Evaluation split', 'Statistical test', 'Analysis scope']


## 11. Integrity checks


In [ ]:
# Harmonised validation results
# Combines Phase 2 Gemma val scores, Phase 3 XLM-R + SERENGETI val scores,
# and Phase 4 XLM-R Gemma + SERENGETI test ablation results (all validation-selected).

LANG_ORDER_H = ["eng","hin","rus","hau","kin","sun","yor","vmw","pcm"]
LANGUAGES_H  = {
    "eng": "English",   "hin": "Hindi",     "rus": "Russian",
    "hau": "Hausa",     "kin": "Kinyarwanda","sun": "Sundanese",
    "yor": "Yoruba",    "vmw": "Emakhuwa",  "pcm": "Nigerian Pidgin"
}
SERENGETI_TARGETS = ["hau","kin","yor","vmw","pcm"]

P2_RESULTS_PATH     = f"{VALID_CKPT}/Phase2/phase2_validation_raw_results_gemma4.json"
P3_XLMR_PATH        = f"{VALID_CKPT}/Phase3/phase3_xlmr_validation_scores.json"
P3_SRNG_PATH        = f"{VALID_CKPT}/Phase3/phase3_serengeti_validation.json"
P4_SRNG_RESULTS     = f"{VALID_CKPT}/Phase4/phase4_serengeti_validation_raw_results.json"
P4_SRNG_FEWSHOT     = f"{VALID_CKPT}/Phase4/phase4_serengeti_fewshot_only_validation_results.json"

p2_r    = load_json(P2_RESULTS_PATH) if os.path.exists(P2_RESULTS_PATH) else {}
p3_xlmr = load_json(P3_XLMR_PATH)   if os.path.exists(P3_XLMR_PATH)   else {}
p3_srng = load_json(P3_SRNG_PATH)   if os.path.exists(P3_SRNG_PATH)   else {}
p4_xlmr = load_json(RESULTS_PATH)   if os.path.exists(RESULTS_PATH)    else {}
p4_srng = load_json(P4_SRNG_RESULTS) if os.path.exists(P4_SRNG_RESULTS) else {}
p4_srng_fs = load_json(P4_SRNG_FEWSHOT) if os.path.exists(P4_SRNG_FEWSHOT) else {}

def _bmark(lang):
    return BENCHMARK_A.get(lang), BENCHMARK_C.get(lang)

rows = []
for lang in LANG_ORDER_H:
    # Phase 2 - Gemma P0-P3 on validation split
    for prompt in ["p0","p1","p2","p3"]:
        if lang in p2_r and prompt in p2_r[lang]:
            _ba, _bc = _bmark(lang)
            rows.append({"Language code": lang.upper(), "System": f"G4-{prompt.upper()}",
                         "Evaluation split": "validation", "Macro-F1": p2_r[lang][prompt]["macro_f1"],
                         "Published Track A test best macro-F1 (context only)": _ba,
                         "Published Track C test best macro-F1 (context only)": _bc,
                         "Benchmark comparison status": BENCHMARK_STATUS})

    # Phase 3 - XLM-R all configs on validation split
    if lang in p3_xlmr:
        for cfg, score in p3_xlmr[lang].items():
            _ba, _bc = _bmark(lang)
            rows.append({"Language code": lang.upper(), "System": f"P3-XLM-A-{cfg}",
                         "Evaluation split": "validation", "Macro-F1": score["macro_f1"],
                         "Published Track A test best macro-F1 (context only)": _ba,
                         "Published Track C test best macro-F1 (context only)": _bc,
                         "Benchmark comparison status": BENCHMARK_STATUS})

    # Phase 3 - SERENGETI track_a configs on validation split
    if lang in p3_srng:
        for key, score in p3_srng[lang].items():
            if key.startswith("track_a_"):
                cfg = key.replace("track_a_","")
                _ba, _bc = _bmark(lang)
                rows.append({"Language code": lang.upper(), "System": f"P3-SRNG-A-{cfg}",
                             "Evaluation split": "validation", "Macro-F1": score["macro_f1"],
                             "Published Track A test best macro-F1 (context only)": _ba,
                             "Published Track C test best macro-F1 (context only)": _bc,
                             "Benchmark comparison status": BENCHMARK_STATUS})

    # Phase 4 - XLM-R Gemma 4-condition ablation on test (validation-selected)
    COND_LABEL = {"neither":"P4-G4-neither","prompt_only":"P4-G4-prompt",
                  "config_only":"P4-G4-config","both":"P4-G4-CAST"}
    if lang in p4_xlmr:
        for cond, score in p4_xlmr[lang].items():
            _ba, _bc = _bmark(lang)
            rows.append({"Language code": lang.upper(), "System": COND_LABEL.get(cond, cond),
                         "Evaluation split": "test_valselected", "Macro-F1": score["macro_f1"],
                         "Published Track A test best macro-F1 (context only)": _ba,
                         "Published Track C test best macro-F1 (context only)": _bc,
                         "Benchmark comparison status": BENCHMARK_STATUS})

    # Phase 4 - SERENGETI 4-condition ablation on test (validation-selected)
    SRNG_COND_LABEL = {"neither":"P4-SRNG-neither","prompt_only":"P4-SRNG-prompt",
                       "config_only":"P4-SRNG-FEWSHOT-ONLY","both":"P4-SRNG-CAST"}
    if lang in p4_srng:
        for cond, score in p4_srng[lang].items():
            _ba, _bc = _bmark(lang)
            rows.append({"Language code": lang.upper(), "System": SRNG_COND_LABEL.get(cond, cond),
                         "Evaluation split": "test_valselected", "Macro-F1": score["macro_f1"],
                         "Published Track A test best macro-F1 (context only)": _ba,
                         "Published Track C test best macro-F1 (context only)": _bc,
                         "Benchmark comparison status": BENCHMARK_STATUS})

harmonised_df   = pd.DataFrame(rows)
harmonised_path = f"{P4_DIR}/phase4_validation_harmonised_results.csv"
harmonised_df.to_csv(harmonised_path, index=False)
print(f"✓ Harmonised validation results saved: {harmonised_path}")
print(f"  Rows: {len(harmonised_df)}  |  Languages covered: {harmonised_df['Language code'].nunique()}")
print(f"  Systems: {sorted(harmonised_df['System'].unique())}")


✓ Harmonised validation results saved: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase4/phase4_validation_harmonised_results.csv
  Rows: 150  |  Languages covered: 9
  Systems: ['G4-P0', 'G4-P1', 'G4-P2', 'G4-P3', 'P3-SRNG-A-C1', 'P3-SRNG-A-C2', 'P3-SRNG-A-C3', 'P3-XLM-A-C1', 'P3-XLM-A-C2', 'P3-XLM-A-C3', 'P3-XLM-A-track_a_C1', 'P3-XLM-A-track_a_C2', 'P3-XLM-A-track_a_C3', 'P3-XLM-A-track_c_C1', 'P3-XLM-A-track_c_C2', 'P3-XLM-A-track_c_C3', 'P4-G4-CAST', 'P4-G4-config', 'P4-G4-neither', 'P4-G4-prompt']


In [ ]:
# Best system per language (validation-selected)
# Includes both XLM-R Gemma and SERENGETI as candidate systems.

frozen_xlmr = load_json(SELECTION_PATH)
srng_sel_path = f"{P4_DIR}/phase4_serengeti_validation_selections.json"
frozen_srng = load_json(srng_sel_path) if os.path.exists(srng_sel_path) else {}

best_rows = []
for lang in LANG_ORDER_H:
    lang_name = LANGUAGES_H.get(lang, lang)

    # XLM-R Gemma selections
    sel_x   = frozen_xlmr["languages"].get(lang, {})
    bp_xlmr = sel_x.get("best_prompt","?")
    bc_xlmr = sel_x.get("best_config","?")

    # SERENGETI selections (only for 5 target languages)
    sel_s   = frozen_srng.get("languages", {}).get(lang, {}) if frozen_srng else {}
    bp_srng = sel_s.get("best_prompt","?") if sel_s else None
    bc_srng = sel_s.get("best_config","?") if sel_s else None

    # Scores
    gemma_p_score  = p2_r.get(lang,{}).get(bp_xlmr.lower(),{}).get("macro_f1")
    xlmr_c_score   = p3_xlmr.get(lang,{}).get(bc_xlmr,{}).get("macro_f1")
    cast_g_score   = p4_xlmr.get(lang,{}).get("both",{}).get("macro_f1")
    cast_srng_score = p4_srng.get(lang,{}).get("both",{}).get("macro_f1") if lang in SERENGETI_TARGETS else None

    # Best overall on test
    candidates = {"G4-prompt-only": p4_xlmr.get(lang,{}).get("prompt_only",{}).get("macro_f1"),
                  "G4-CAST(XLM-R)": cast_g_score,
                  "SRNG-CAST":       cast_srng_score}
    candidates = {k: v for k, v in candidates.items() if v is not None}
    if candidates:
        best_sys = max(candidates, key=candidates.get)
        best_f1  = candidates[best_sys]
    else:
        best_sys, best_f1 = "?", None

    best_rows.append({
        "Language":                                                       lang_name,
        "Language code":                                                  lang.upper(),
        "Validation-selected prompt for XLM-R route":                    bp_xlmr,
        "Validation-selected XLM-R configuration":                       bc_xlmr,
        "Validation-selected SERENGETI configuration":                   bc_srng if bc_srng else None,
        "Best XLM-R validation macro-F1":                                xlmr_c_score,
        "Gemma CAST test macro-F1 using XLM-R-selected configuration":   cast_g_score,
        "Gemma CAST test macro-F1 using SERENGETI-selected configuration": cast_srng_score,
        "Highest-scoring Phase 4 condition":                             best_sys,
        "Highest Phase 4 test macro-F1":                                 best_f1,
        "Published Track A test best macro-F1 (context only)":           BENCHMARK_A.get(lang),
        "Published Track C test best macro-F1 (context only)":           BENCHMARK_C.get(lang),
        "Benchmark comparison status":                                    BENCHMARK_STATUS,
    })

best_df   = pd.DataFrame(best_rows)
best_path = f"{P4_DIR}/phase4_validation_best_system.csv"
best_df.to_csv(best_path, index=False)
print(f"✓ Best system per language saved: {best_path}")
print("\n" + "=" * 80)
print(f"{'Language':<18} {'G4-CAST':>9} {'SRNG-CAST':>10}  Best")
print("-" * 80)
for _, row in best_df.iterrows():
    g4   = f"{row['Gemma CAST test macro-F1 using XLM-R-selected configuration']:.4f}"  if row['Gemma CAST test macro-F1 using XLM-R-selected configuration']  is not None else "  --- "
    srng = f"{row['Gemma CAST test macro-F1 using SERENGETI-selected configuration']:.4f}" if row['Gemma CAST test macro-F1 using SERENGETI-selected configuration'] is not None else "  --- "
    f1   = f"{row['Highest Phase 4 test macro-F1']:.4f}"        if row['Highest Phase 4 test macro-F1']        is not None else "  --- "
    print(f"{row['Language']:<18} {g4:>9} {srng:>10}  {row['Highest-scoring Phase 4 condition']} ({f1})")
print("=" * 80)


✓ Best system per language saved: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase4/phase4_validation_best_system.csv

Language             G4-CAST  SRNG-CAST  Best
--------------------------------------------------------------------------------
English               0.6290       ---   G4-CAST(XLM-R) (0.6290)
Hindi                 0.8496       ---   G4-CAST(XLM-R) (0.8496)
Russian               0.8559       ---   G4-CAST(XLM-R) (0.8559)
Hausa                 0.5741       ---   G4-prompt-only (0.5790)
Kinyarwanda           0.4347       ---   G4-prompt-only (0.4417)
Sundanese             0.5212       ---   G4-prompt-only (0.5596)
Yoruba                0.3105       ---   G4-prompt-only (0.3202)
Emakhuwa              0.0980       ---   G4-CAST(XLM-R) (0.0980)
Nigerian Pidgin       0.5650       ---   G4-CAST(XLM-R) (0.5650)
